# Module 00 · Prerequisites

This notebook turns the lesson's small examples into executable checks. Run the cells from top to bottom, then change the probabilities and payoff tables to explore best responses. It uses only Python's standard library.

## Expected value and variance

Represent a lottery as `(probability, payoff)` pairs. Probabilities should sum to one.

In [ ]:
def expected_value(outcomes):
    return sum(probability * payoff for probability, payoff in outcomes)

def variance(outcomes):
    mean = expected_value(outcomes)
    return sum(probability * (payoff - mean) ** 2 for probability, payoff in outcomes)

coin_bet = [(0.5, 3), (0.5, -2)]
print('expected payoff:', expected_value(coin_bet))
print('variance:', variance(coin_bet))
assert expected_value(coin_bet) == 0.5
assert variance(coin_bet) == 6.25

## Matching pennies

The matrix stores player 1's payoff. The column player's payoff is its negative. A strategy is a probability vector over actions.

In [ ]:
matching_pennies = [[1, -1], [-1, 1]]  # rows/columns: Heads, Tails

def row_action_values(payoff_matrix, column_mix):
    return [sum(payoff * probability for payoff, probability in zip(row, column_mix))
            for row in payoff_matrix]

def column_action_values(payoff_matrix, row_mix):
    n_columns = len(payoff_matrix[0])
    return [-sum(row_mix[i] * payoff_matrix[i][j]
                 for i in range(len(payoff_matrix)))
            for j in range(n_columns)]

fair = [0.5, 0.5]
print('row values vs fair column:', row_action_values(matching_pennies, fair))
print('column values vs fair row:', column_action_values(matching_pennies, fair))
assert row_action_values(matching_pennies, fair) == [0.0, 0.0]
assert column_action_values(matching_pennies, fair) == [0.0, 0.0]

In [ ]:
# Against a biased opponent, choose the action with the larger expected payoff.
column_mix = [0.7, 0.3]
values = row_action_values(matching_pennies, column_mix)
print('Heads, Tails values:', values)
print('best response:', ['Heads', 'Tails'][max(range(2), key=values.__getitem__)])
assert abs(values[0] - 0.4) < 1e-12 and abs(values[1] + 0.4) < 1e-12

## Rock-paper-scissors

Rows and columns are ordered Rock, Paper, Scissors. Check that the uniform mixture makes every pure action equally good. Then edit `opponent` to see the best response change.

In [ ]:
rps = [[0, -1, 1], [1, 0, -1], [-1, 1, 0]]
uniform = [1 / 3, 1 / 3, 1 / 3]
values = row_action_values(rps, uniform)
print('Rock, Paper, Scissors values vs uniform:', values)
assert all(abs(value) < 1e-12 for value in values)

opponent = [0.6, 0.2, 0.2]
values = row_action_values(rps, opponent)
actions = ['Rock', 'Paper', 'Scissors']
print('values vs changed mix:', values)
print('best response:', actions[max(range(3), key=values.__getitem__)])

## Card counting, pot odds, and Bayes

The shared helpers live in `pokerbot.foundations`; run this notebook from the repository root.

In [ ]:
from pathlib import Path
import sys

for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / "pokerbot").is_dir():
        sys.path.insert(0, str(parent))
        break
else:
    raise RuntimeError("Run this notebook inside the pokerbots repository")

from pokerbot.foundations import (
    bayes_posterior, bluff_break_even_fold_rate, call_equity_required,
    combinations_remaining, minimum_defense_frequency, rank_combos,
)

print("Starting hands:", combinations_remaining(52, 0))
print("Opponent hands after two known cards:", combinations_remaining(52, 2))
print("Pocket aces after holding one ace:", rank_combos(4, known_of_rank=1))
print("Equity needed to call 100 into 100 after a 100 bet:", call_equity_required(100, 100))
print("Fold frequency for a pot-sized bluff:", bluff_break_even_fold_rate(100, 100))
priors = [0.2, 0.5, 0.3]
bet_likelihoods = [0.8, 0.4, 0.1]
print("Posterior after a bet:", bayes_posterior(priors, bet_likelihoods))
assert combinations_remaining(52, 0) == 1326
assert combinations_remaining(52, 2) == 1225
assert rank_combos(4, known_of_rank=1) == 3
assert abs(call_equity_required(100, 100) - 1/3) < 1e-12
assert abs(bluff_break_even_fold_rate(100, 100) - 0.5) < 1e-12

## Solve a matrix game two ways

The small-game library has a support-indifference solver and a primal/dual minimax LP solver. Compare their answers on matching pennies, RPS, and a rectangular game.

In [ ]:
from pathlib import Path
import sys

for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / "pokerbot").is_dir():
        sys.path.insert(0, str(parent))
        break
else:
    raise RuntimeError("Run this notebook inside the pokerbots repository")

from pokerbot.foundations import solve_zero_sum, solve_zero_sum_lp

games = {
    "matching pennies": [[1, -1], [-1, 1]],
    "rock-paper-scissors": rps,
    "rectangular example": [[3, 1, 2], [0, 2, 1]],
}
for name, matrix in games.items():
    by_indifference = solve_zero_sum(matrix)
    by_lp = solve_zero_sum_lp(matrix)
    print(name)
    print("  indifference:", by_indifference)
    print("  LP:", by_lp)
    assert abs(by_indifference[2] - by_lp[2]) < 1e-8

## A tiny tree and its normal form

Player 0 acts at the root; Player 1 then chooses at either of two separate decision nodes. The tree has 7 nodes (1 root, 2 decisions, 4 leaves), while the normal-form strategies must specify one action for each decision node.

In [ ]:
from itertools import product

actions = ("left", "right")
p0_plans = list(product(actions, repeat=1))
p1_plans = list(product(actions, repeat=2))
print("Tree nodes:", 1 + 2 + 4)
print("Player 0 complete plans:", p0_plans)
print("Player 1 complete plans:", p1_plans)
print("Normal-form strategy profiles:", len(p0_plans) * len(p1_plans))
assert len(p0_plans) == 2 and len(p1_plans) == 4

## Try it yourself

1. Change the matching-pennies mix to `[0.5, 0.5]`; confirm neither action can gain.
2. Change one RPS probability and identify a best response by hand before running the cell.
3. Change matching pennies to pay `+2` for a match and `−1` for a mismatch. What probability of Heads makes the other player indifferent?
4. Explain why a payoff table cannot express the order of a poker hand's actions.